# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a-h-taju/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I chose **Lane 2: Refresh / Content Opportunity Scoring**.

The goal is to investigate which content pages should be reviewed first for a potential refresh, expansion, protection, pruning, or monitoring action.

This lane connects observable content and search-performance signals to a practical decision: deciding which pages deserve limited human review capacity first.

I will begin with the anonymized starter dataset. I want to investigate whether multiple observable signals can support a useful ranking of review candidates rather than relying only on a single hand-written rule.

The lane is provisional and may change if later data analysis shows that another research direction is better supported by the available evidence.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

### Research Question

**Which content pages should be reviewed first for a potential refresh, expansion, protection, pruning, or monitoring action based on observable content and search-performance signals?**

### Decision

The decision is:

**Which pages should receive limited review capacity first?**

### Who Acts on the Output?

A content or SEO editor would act on the output by reviewing the highest-priority pages and deciding whether they need an appropriate action.

### Action

The reviewer may decide to:

- refresh outdated content,
- expand useful content,
- protect pages that are performing well,
- investigate possible consolidation or pruning,
- or monitor a page instead of changing it.

The system is intended to prioritize pages for human review, not to automatically make the final content decision.

### Unit of Analysis

The unit of analysis is **one pseudonymized content page**.

### Output

The intended output is a **ranked review queue** with a priority score and understandable reason codes showing which observable signals contributed to the prioritization.

### Task Type

This is primarily a **ranking / scoring** problem because the practical question is "Which ones first?"

### Target

The initial starter-data target/proxy is `is_declining_label`, which is derived from `trend_direction`.

Because this label is rule-derived from the current trend window rather than a future observed outcome, I will treat it as a proxy for initial exploration rather than as a perfect future target.

### Success Metric

The primary ranking metric I would use is **Precision@K**, where K represents a realistic number of pages that a reviewer can inspect.

I may also report Average Precision (AP) and ROC-AUC as supporting metrics.

### Cost of a Wrong Call

A false positive could waste editor or reviewer time on a page that did not need attention.

A false negative could cause a potentially useful review opportunity to be missed.

Therefore, the ranking should prioritize useful candidates while keeping the reasons understandable.

### Why Data or ML Helps

A simple rule may be useful as a baseline, but multiple signals such as visibility, traffic, content age, search performance, and engagement may interact in ways that are difficult to capture with one hand-written rule.

ML may be useful if it improves the ranking of review candidates beyond a simple baseline. If a simple rule performs adequately, a more complex model may not be necessary.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir())

Current working directory:
/content

Files/folders here:
['.config', 'sample_data']


In [4]:
import os

for root, dirs, files in os.walk("/content"):
    if "content_refresh_anonymized.csv" in files:
        print("FOUND:")
        print(os.path.join(root, "content_refresh_anonymized.csv"))

In [5]:
!git clone https://github.com/a-h-taju/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 126 (delta 36), reused 97 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (126/126), 1.87 MiB | 6.87 MiB/s, done.
Resolving deltas: 100% (36/36), done.


In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset loaded successfully.
Rows: 30000
Columns: 44


In [7]:
required_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "trend_direction"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    print("Missing required columns:", missing_columns)
else:
    print("All required columns are present.")

All required columns are present.


In [8]:
print("Unique content pages:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())
print("Median content age:", round(df["content_age_days"].median(), 2), "days")

Unique content pages: 30000
Unique clients: 32
Median content age: 236.0 days


In [9]:
trend_counts = df["trend_direction"].value_counts(dropna=False)

print("Trend direction counts:")
print(trend_counts)

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [10]:
down_pct = (df["trend_direction"].eq("down").mean() * 100)

print(f"Pages with trend_direction = 'down': {down_pct:.2f}%")

Pages with trend_direction = 'down': 54.21%


In [11]:
print("Median impressions over 90 days:",
      round(df["impressions_90d"].median(), 2))

print("Median sessions over 90 days:",
      round(df["sessions_90d"].median(), 2))

Median impressions over 90 days: 731.0
Median sessions over 90 days: 7.0


The starter dataset contains 30,000 unique content pages across 32 clients.

The median content age is 236 days, showing that the dataset includes content with a range of freshness levels. The median number of impressions over the last 90 days is 731, while the median number of sessions is 7.

In addition, 54.21% of the pages have trend_direction = "down" in the starter dataset. This provides a useful starting point for investigating which observable content and search-performance signals are associated with pages that may deserve review.

These numbers support the Lane 2 research question because the dataset contains a large set of content pages with different levels of age, visibility, traffic, and observed trend direction. However, trend_direction and the derived is_declining_label should be treated as a current-window proxy rather than proof of future decline or evidence that refreshing a page will cause its performance to recover.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What I Can Claim

I can investigate whether observable content and search-performance signals are associated with pages that may deserve review.

I can compare a transparent baseline with a machine-learning approach and evaluate whether the ranking improves under a defined metric such as Precision@K.

I can identify patterns in the available data and use them to support human review prioritization.

### What I Cannot Claim

I cannot claim that Google uses a particular feature as a ranking factor based on this dataset.

I cannot claim that refreshing a page will definitely cause its performance to recover.

I cannot interpret an association as proof of causation.

I cannot use `health_score`, `priority_score`, or `action_type` as ordinary model features or targets because these represent product decisions rather than independent observable signals.

I also cannot treat `is_declining_label` as a perfect future outcome because it is derived from the current `trend_direction` information.

Therefore, I will describe the results as observational and decision-support evidence rather than causal proof or guaranteed outcomes.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- [x] I selected a provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.
- [x] I defined a concrete research question.
- [x] I identified the unit of analysis.
- [x] I identified the decision that the output supports.
- [x] I identified who acts on the output.
- [x] I described the possible actions.
- [x] I explained the cost of a wrong recommendation.
- [x] I explained why data or ML may help.
- [x] I identified ranking/scoring as the task type.
- [x] I named Precision@K as the primary success metric.
- [x] I inspected the starter dataset using code.
- [x] I included actual numbers from the dataset.
- [x] I will not use content_id or client_id as model features.
- [x] I will not use trend_direction or trend_pct as model features.
- [x] I understand that is_declining_label is a proxy rather than a perfect future outcome.
- [x] I will avoid causal claims.
- [x] I understand that the lane is provisional and can change later.